## Setup

1. Clone & cd into this repo.
2. Start containers as [shown here](https://github.com/josephmachado/airflow-tutorial?tab=readme-ov-file#setup).
3. Open [airflow-tutorial.ipynb](https://github.com/josephmachado/airflow-tutorial/blob/main/notebooks/airflow-tutorial.ipynb) locally.

## Data Pipelines Interact With Multiple Systems

* Data pipelines often involve running **tasks on multiple systems in a specific order**. E.g., Extract data from API, DBs, sending emails, etc., aka Orchestrations
* Data pipelines need to run on some **schedule**. E.g., Hourly, Every Tuesday, etc., aka Scheduling
* Data teams need to schedule and orchestrate their pipelines. 
* Data teams need **observability** of their pipelines. i.e. are my pipelines running as expected? Why is this pipeline taking so long?, etc
* Apache Airflow was developed to address these requirements.
* *Alternatives*: dbt-core (orchestrate), dbt-cloud (orchestrate & schedule), cronjob (schedule), Native Python (custom build orchestrate), Dagster, Prefect, etc

![Sample Batch Pipeline](batch_pipeline.svg)

## Create Pipelines With Apache Airflow

* Apache Airflow runs the tasks of your pipeline in the order you define.

* In Airflow a pipeline is defined with a `DAG (Directed Acyclic Graph)` which is a one-way graph with a defined end. 

* Every DAG is made up of one or more tasks. Each `task` represents a unit of work in your pipeline.

* You can design a task to be whatever you want it to be. Airflow has 3 types of tasks:

    - `Operators` are prebuilt tasks for specific operations. E.g., S3CreateBucketOperator, SQLExecuteQueryOperator. You can see a list of all the [active operators here](https://airflow.apache.org/docs/apache-airflow-providers/operators-and-hooks-ref/index.html).
    - `Sensors` are tasks that keep checking for an event. A sensor will periodically poke the external system to see if an event occurred. E.g. [S3KeySensor](https://airflow.apache.org/docs/apache-airflow-providers-amazon/stable/operators/s3/s3.html#wait-on-an-amazon-s3-key)
    -  Python function that can be used as a task by using the `@task` decorator.
  

#### Example

[Click here](../airflow/dags/basic_example.py)

* There are 2 key things happening here 

  - `Scheduling`: We schedule the DAG to run daily. We use the @daily preset. But we can use any valid cron schedule. 

  - `Orchestration`: We define the order in which tasks are to be run. The extract task runs first, and its output is sent to the transform task, which then sends its output to the load task (aka `task dependency`).

* We also define DAG and Task attributes. Such as if the DAG should catch up from its start date, or if the transform tasks need to be retried. 

* The dag and task decorator have multiple options that you can set. If you need to implement custom orchestration logic, start by reviewing the DAG and task attributes. E.g., Functions to be called on failure/Success, DAG timeout, etc.

In [1]:
from airflow.sdk import DAG
DAG?

2026-04-11T17:14:46.584134Z [info     ] setup plugin alembic.autogenerate.schemas [alembic.runtime.plugins] loc=plugins.py:37
2026-04-11T17:14:46.585468Z [info     ] setup plugin alembic.autogenerate.tables [alembic.runtime.plugins] loc=plugins.py:37
2026-04-11T17:14:46.586549Z [info     ] setup plugin alembic.autogenerate.types [alembic.runtime.plugins] loc=plugins.py:37
2026-04-11T17:14:46.589531Z [info     ] setup plugin alembic.autogenerate.constraints [alembic.runtime.plugins] loc=plugins.py:37
2026-04-11T17:14:46.590384Z [info     ] setup plugin alembic.autogenerate.defaults [alembic.runtime.plugins] loc=plugins.py:37
2026-04-11T17:14:46.591294Z [info     ] setup plugin alembic.autogenerate.comments [alembic.runtime.plugins] loc=plugins.py:37


Init signature:
DAG(
    dag_id: 'str',
    *,
    description: 'str | None' = None,
    default_args=NOTHING,
    start_date: 'datetime | None' = NOTHING,
    end_date: 'datetime | None' = None,
    schedule: 'ScheduleArg' = None,
    template_searchpath: 'str | Iterable[str] | None' = None,
    template_undefined: 'type[jinja2.StrictUndefined]' = <class 'jinja2.runtime.StrictUndefined'>,
    user_defined_macros: 'dict | None' = None,
    user_defined_filters: 'dict | None' = None,
    max_active_tasks=NOTHING,
    max_active_runs=NOTHING,
    max_consecutive_failed_dag_runs=NOTHING,
    dagrun_timeout: 'timedelta | None' = None,
    deadline: 'list[DeadlineAlert] | DeadlineAlert | None' = None,
    sla_miss_callback: 'None' = None,
    catchup: 'bool' = NOTHING,
    on_success_callback: 'None | DagStateChangeCallback | list[DagStateChangeCallback]' = None,
    on_failure_callback: 'None | DagStateChangeCallback | list[DagStateChangeCallback]' = None,
    doc_md: 'str | None' = None,


In [2]:
from airflow.sdk import BaseOperator 
# This is what @task uses underneath
BaseOperator?

Init signature:
BaseOperator(
    *,
    task_id: 'str',
    owner: 'str' = 'airflow',
    email: 'str | Sequence[str] | None' = None,
    email_on_retry: 'bool' = True,
    email_on_failure: 'bool' = True,
    retries: 'int | None' = 0,
    retry_delay: 'timedelta | float' = datetime.timedelta(seconds=300),
    retry_exponential_backoff: 'bool' = False,
    max_retry_delay: 'timedelta | float | None' = None,
    start_date: 'datetime | None' = None,
    end_date: 'datetime | None' = None,
    depends_on_past: 'bool' = False,
    ignore_first_depends_on_past: 'bool' = False,
    wait_for_past_depends_before_skipping: 'bool' = False,
    wait_for_downstream: 'bool' = False,
    dag: 'DAG | None' = None,
    params: 'collections.abc.MutableMapping[str, Any] | None' = None,
    default_args: 'dict | None' = None,
    priority_weight: 'int' = 1,
    weight_rule: 'str | PriorityWeightStrategy' = <WeightRule.DOWNSTREAM: 'downstream'>,
    queue: 'str' = 'default',
    pool: 'str | None' = No

#### Exercise [10 min]

* Create a DAG called minute_etl that runs every minute and has 2 tasks (task_a, task_b) that are run in parallel after which a task_c is finally run. 

* These task_* are Python functions that just print the name of the task (hardcode 1 function per task).

* Create a python script at `$AIRFLOW_HOME/dags` folder.

![Parallel Pipeline](parallel_pipeline.svg)

In [ ]:
! echo $AIRFLOW_HOME/dags

In [ ]:
[task_a(), task_b()] >> task_c()

#### Exercise [5 min]

* Assume you have a pipeline that takes 10h to run and the start date is 2025-01-01 (today is 2026-04-11). Will you set the DAG with `catchup = True` or `catchup = False`?



## Data Pipelines Are Typically Full-Refresh Or Incremental 

* Most data pipelines are either Snapshot or Incremental.

  - `Snapshot` (aka full-refresh) pipeline processes the entire source data with each run 

  - `Incremental` pipeline processes source data filtered per time range 

![Snapshot & Incremental Pipelines](snapshot_v_incremental.svg)

* *Note*: Processing past n days of data, processing a lookback window, etc are variants of incremental pipelines.

#### Example

* Let’s take a look at an example incremental pipeline, [standard_data_interval_example](../airflow/dags/standard_data_interval_example.py).

* Open this DAG and run it at, [minutely_interval_printer](http://localhost:8080/dags/minutely_interval_printer).

![Data Time Interval](data_interval.png)

* In this pipeline we see how we schedule using a [CronDataIntervalTimetable](https://airflow.apache.org/docs/apache-airflow/stable/authoring-and-scheduling/timetable.html#crondataintervaltimetable) method.

* The [CronDataIntervalTimetable](https://airflow.apache.org/docs/apache-airflow/stable/authoring-and-scheduling/timetable.html#crondataintervaltimetable) method allows us to indicate the time range of source to be processed. 

* *Note*: The IntervalTimetable **equates time range (start and end times) with the frequence of the pipeline**. This cannot be used if the time range to be processed overlaps with the frequency. 

![Overlapping Data Time Range](overlapping_data_interval.png)

* In cases where the time range to be processed overlaps with the schedule, use a `trigger + interval pattern`.

#### Exercise [10 min]

* Create a pipeline called `custom_interval` which runs every minute, but processes a days worth of data with each run.

* Similar to the example above you just need to print the intervals.

* Use [custom_data_interval_minutely_interval_printer](../airflow/dags/custom_data_interval_example.py) as an example to see how to use `TriggerTimetable.`

* We have used the Cron data interval and trigger timetables. Cron schedules usually run at the start of some schedule. 

* We can also use `Delta` data interval and trigger timetables. Add links. Delta starts counting from when you actually turn on the DAG, whereas Cron snaps to clock time.

| | Cron (`*/30 * * * *`) | Delta (30 min) |
|---|---|---|
| DAG start | 12:38 PM | 12:38 PM |
| 1st run | 1:00 PM | 1:08 PM |
| 2nd run | 1:30 PM | 1:38 PM |
| 3rd run | 2:00 PM | 2:08 PM |
| Data window | Fixed clock intervals | Relative to last run |
| Misses 12:38–1:00 PM? | ✅ Yes | ❌ No |


## Coordinating multiple pipelines is tricky; use asset-based scheduling instead

* As your data pipeline grow in complexity you will run into cross DAG dependency. That is, a DAG (aka downstream) depends on another (aka upstream) completing.

* Companies used a variety of patterns to deal with this 

  - `Sensor` task in the downstream DAG. The problem is that this DAG can wait indefinitely, and that would need to be handled.

  - Estimating when the upstream DAG will complete and adding a buffer (say 1h) to the downstream DAG. This approach is unreliable.

  - Re-running the downstream DAG multiple times to hope to catch the upstream updates. Unreliable and expensive.

  - Use `TriggerDagRunOperator` task in the upstream DAG to start the downstream DAG.

* In recent years, orchestration tools have starting incorporating **data update based scheduling**. 

* With data update based scheduling an upstream DAG informs Airflow that it has updated a specific dataset and Airflow identifies all the downstream DAGs that are waiting for this update and starts them.

* In Airflow, this is called `data asset based scheduling.` 

#### Example 

* Let’s look at a simple example of a downstream and upstream DAGs that coordinate using data asset based scheduling.


* [Upstream DAG](../airflow/dags/asset_producer_example.py)
* [Downstream DAG](../airflow/dags/asset_consumer_example.py)

![Data asset UI](data_asset.png)
  
* Let’s dissect the process:
    * The `asset` is identified by a unique url
    * A `task outlet` tells Airflow that the specified outlet asset is updated (irrespective of the actual processing)
    * The consumer DAG will use the asset as a schedule
    * The `triggering_asset_event` will be filled in by Airflow and have the information about the asset and the DAG that updated it

#### Exercise [10 min]

* Create a pipeline `simple_data_pipeline` that runs every minutes.
* In `simple_data_pipeline` create a task to indicate update to a data asset `sample_data_asset`.
* Use [Upstream DAG](../airflow/dags/asset_producer_example.py) as a prototype.

* Create a downstream pipeline called `new_downstream` similar to [Downstream DAG](../airflow/dags/asset_consumer_example.py).
* But instead of just depending on `hourly_report` asset, it should also depend on `sample_data_asset`

**Note** You should see this in your UI

![Multi Asset](multi_asset.png)

## Airflow’s Dag Processor Creates DAGs, and the Scheduler Starts Them

When you create a Python file with a DAG definition, the

* `dag-processor` is a Python process responsible for parsing python files with DAG definitions and creating  DAGs for them. 

* The DAG information is stored in Airflow’s database.

* The `scheduler` process checks (by default every minute) whether any DAGs need to be started, and starts it.

* Let’s take a look at these processes as shown below.

In [ ]:
! ps aux | grep '[d]ag-processor' # You will see the dag-processor process 

In [ ]:
! ps aux | grep '[a]irflow scheduler' # You will see the scheduler process

* Airflow tasks are run in parallel using the executors.

* We use the LocalExecutor, which runs tasks as individual Python processes.

In [ ]:
! ps aux | grep '[a]irflow worker' | wc -l # This will show 32 workers

In [ ]:
!cat $AIRFLOW_HOME/airflow.cfg | grep 'parallelism =' 
# airflow.cfg Contains Airflows Configuration and this Will Shown Parallelism = 32 Corresponding to the 32 We See From the Previous Command

[airflow.cfg](../airflow/airflow.cfg)

## Best Practices

* Use Airflow for scheduling and orchestration, keep **logic in individual scripts**. This ensures single responsibility, making debugging and maintenance simple.

* Airflow **tasks should trigger the process and not actually process the data**. E.g., Airflow tasks trigger Spark, Snowflake jobs, etc.

* The exception is If you are using dedicated VMs per task (e.g. Celery or k8s executors) processing in Airflow tasks should be fine. 

* Do not write high level code, ie. do not perform heavy code (e.g. Creating a SparkSession) outside the Task as the dag-processor will rerun the script multiple times during parsing.

* Executor are systems which define how your tasks are run. You can get far with LocalExecutor & a machine with many cores.

* Use xcom for passing small data, for large data use an intermediate cloud storage system (S3 or similar)

## Airflow is powerful, don’t create your own orchestrator

* While Airflow may feel clunky, when used with the best practices above its pretty good. 

* Here are some features which may seem basic, but are incredibly powerful for what they enable.

  - Check state, history, logs, code, run backfills, etc via the UI

  - Variables/Configs/Connections set via UI 

  - RBAC for role-based access to monitor/edit DAGs

  - Airflow CLI & API to work with DAGs & Tasks

## Go from writing code to designing systems.

**Data Engineering Course — Opens April 26th**

What you'll learn:

* Data Warehouse design & Medallion Architecture
* Pipeline Design Patterns & Data Quality
* Orchestration & Scheduling Patterns
* Spark API & Storage Optimization
* Data Processing & Code Optimizations
* Interview Prep & 2 Capstone Projects

The Data Engineering Course is where it all comes together. This course helps you move from writing code to designing systems.

What’s included:

* 10 hours of self-paced video
* 32 hours of live office hours
* 2 Capstone projects using real StackOverflow data

Newsletter subscribers get an early bird **50% discount (~999~ → $499)**

**75 spots only**
